# Drop input on LSD: with control vs no control (no training)

For each LSD checkpoint in `checkpoints/lsd_*.pt`, evaluate the full metric suite twice — once with each subject's true control vector and once with `control=None` — split by condition (Placebo, LSD, LSD+KET). Plots a 3-panel paired bar chart per model: grey = no input, colored = input. Placebo (control=(0,0)) is the negative control: the model should give nearly identical results with vs without input there. The real test of whether the model uses the input is on LSD and LSD+KET. Uses the same nine metrics as section 4 of `empirical_vs_simulated_comparison.ipynb`.

In [1]:
from pathlib import Path
import json
import sys
import numpy as np
import torch
import matplotlib.pyplot as plt

project_root = Path.cwd()
if not (project_root / 'src').exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.dataset import NeuroscienceDataset, create_data_loaders
from src.models import load_model_from_checkpoint
from src.training import HopfConfig
from src.training.evaluation import (
    EVAL_METRIC_KEYS,
    MetricAccumulator,
    accumulate_timeseries_metrics,
    build_eval_metrics,
    rollout_model,
)
from src.utils import resolve_device, seed_all

plt.rcParams['figure.dpi'] = 140

In [2]:
DEVICE = resolve_device('auto')
seed_all(42)

MODELS = {                                      # checkpoint stem -> (display label, color)
    'lsd_hopf':           ('Coupled Hopf', '#1f77b4'),
    'lsd_nsde':           ('Neural SDE',   '#d62728'),
    'lsd_hybrid_hopf':    ('Hybrid Hopf',  '#2ca02c'),
    'lsd_gnn_hopf':       ('GNN Hopf',     '#9467bd'),
    'lsd_hybrid_neural':  ('Hopf+Neural',  '#ff7f0e'),
}

# (display_label, metric_key, higher_is_better)
METRICS = [
    ('FC \u2191',     'fc_correlation',          True),
    ('FC MSE \u2193', 'fc_mse',                  False),
    ('phFC \u2191',   'phase_fc_correlation',    True),
    ('FCD \u2193',    'fcd_ks',                  False),
    ('phFCD \u2193',  'phfcd_ks',                False),
    ('Meta \u2193',   'metastability_diff',      False),
    ('TS \u2191',     'temporal_correlation',    True),
    ('PSD \u2193',    'power_spectrum_distance', False),
    ('Auto \u2193',   'autocorr_distance',       False),
]

NO_INPUT_COLOR = '#9a9a9a'
OUT_BASE = project_root / 'paper'
for sub in ['images/comparison', 'images_png/comparison', 'images_svg/comparison']:
    (OUT_BASE / sub).mkdir(parents=True, exist_ok=True)

Using CPU


In [3]:
# Use the same defaults the training pipeline uses, so split + window_size match.
lsd_data_dir = str(project_root / 'data' / 'lsd')
cfg = HopfConfig(dataset_type='lsd', lsd_data_dir=lsd_data_dir, use_wandb=False)
dataset = NeuroscienceDataset.from_lsd(
    data_dir=cfg.lsd_data_dir,
    normalize=True,
    device=DEVICE,
    dt=cfg.tr,
    fourier_denoise=cfg.fourier_denoise,
    denoise_f_lo=cfg.denoise_f_lo,
    denoise_f_hi=cfg.denoise_f_hi,
)
print(f'subjects={dataset.n_subjects}  rois={dataset.n_rois}  T={dataset.n_timepoints}  ctrl_dims={dataset.n_control_dims}')

window_size = min(cfg.window_size, dataset.n_timepoints // 2)
_, _, test_inter_loader, test_intra_loader = create_data_loaders(
    dataset=dataset,
    window_size=window_size,
    batch_size=cfg.batch_size,
    n_windows_per_epoch=cfg.n_windows_per_epoch,
    train_ratio=cfg.train_ratio,
    val_ratio=cfg.val_ratio,
    seed=cfg.seed,
    device=DEVICE,
    use_full_timeseries=cfg.use_full_timeseries,
)
EVAL_LOADER = test_inter_loader
print(f'eval batches: {len(EVAL_LOADER)}  window_size={window_size}')

LSD dataset: conditions=['Placebo', 'LSD', 'LSD+Ketanserin'], control_vectors=[(0.0, 0.0), (0.0, 1.0), (1.0, 0.0)], n_patients=25, n_timeseries=75
subjects=75  rois=4  T=240  ctrl_dims=2
  Patient-level split: 17 train / 3 val / 5 test patients  →  51/9/15 timeseries
eval batches: 2  window_size=100


In [4]:
CONDITIONS = [
    ('Placebo', (0.0, 0.0)),
    ('LSD',     (1.0, 0.0)),
    ('LSD+KET', (0.0, 1.0)),
]


def evaluate_per_condition(model, loader, *, drop_control: bool):
    """Evaluate the metric suite separately for each condition.

    Returns ``{condition_name: metrics_dict}``. Rows of every batch are grouped
    by their (LSD, Ketanserin) control vector so each condition gets its own
    accumulator.  When ``drop_control`` is True the model is rolled out with
    ``control=None`` regardless of the row's true condition — Placebo (0, 0)
    will then match the control-coupling-free dynamics exactly.
    """
    eval_modules = build_eval_metrics(
        tr=cfg.tr, fcd_win_sec=cfg.fcd_win_sec, fcd_step_sec=cfg.fcd_step_sec,
    )
    accumulators = {name: MetricAccumulator() for name, _ in CONDITIONS}

    for batch in loader:
        if len(batch) == 4:
            windows, _, _, control = batch
        else:
            windows, _, _ = batch
            control = None
        windows = windows.to(model.device)
        if control is None or control.numel() == 0:
            continue
        control = control.to(model.device)

        for cond_name, cond_vec in CONDITIONS:
            cond_t = torch.tensor(cond_vec, device=control.device, dtype=control.dtype)
            mask = (control == cond_t).all(dim=-1)
            if not mask.any():
                continue
            sub_windows = windows[mask]
            sub_control = control[mask]
            n_steps = sub_windows.shape[2]
            target = sub_windows[:, :, :n_steps]
            ic = target[:, :, 0]
            with torch.no_grad():
                simulated = rollout_model(
                    model, ic, n_steps, cfg.tr,
                    control=None if drop_control else sub_control,
                    sde_type=getattr(cfg, 'sde_type', 'ito'),
                    method=getattr(cfg, 'sde_method', 'euler'),
                    dt_min=getattr(cfg, 'dt_min', 0.1),
                    use_adjoint=getattr(cfg, 'use_adjoint', False),
                    adjoint_method=getattr(cfg, 'adjoint_method', None),
                    denoise_f_lo=getattr(cfg, 'denoise_f_lo', None),
                    denoise_f_hi=getattr(cfg, 'denoise_f_hi', None),
                )
                accumulate_timeseries_metrics(
                    accumulators[cond_name], simulated, target, eval_modules,
                    group_size=getattr(cfg, 'group_size', 0),
                )

    out = {}
    for cond_name, _ in CONDITIONS:
        acc = accumulators[cond_name]
        if not acc.sums:
            out[cond_name] = {}
            continue
        keys = sorted(set(EVAL_METRIC_KEYS) | set(acc.sums.keys()))
        out[cond_name] = acc.summary(keys=keys, include_std=True)
    return out

In [5]:
results = {}  # ckpt_stem -> {cond_name: {'with': metrics, 'without': metrics}}
for ckpt_stem in MODELS:
    ckpt_path = project_root / 'checkpoints' / f'{ckpt_stem}.pt'
    if not ckpt_path.exists():
        print(f'  skipping {ckpt_stem} (no checkpoint at {ckpt_path})')
        continue
    model, model_class, _ = load_model_from_checkpoint(str(ckpt_path), device=DEVICE)
    n_ctrl = int(getattr(model, 'n_control_dims', 0) or 0)
    print(f'\n=== {ckpt_stem}  ({model_class}, n_control_dims={n_ctrl}) ===')

    seed_all(42)
    with_per_cond = evaluate_per_condition(model, EVAL_LOADER, drop_control=False)
    seed_all(42)
    without_per_cond = evaluate_per_condition(model, EVAL_LOADER, drop_control=True)

    results[ckpt_stem] = {
        cond: {'with': with_per_cond.get(cond, {}), 'without': without_per_cond.get(cond, {})}
        for cond, _ in CONDITIONS
    }

    for cond, _ in CONDITIONS:
        w = with_per_cond.get(cond, {}).get('fc_correlation', float('nan'))
        wo = without_per_cond.get(cond, {}).get('fc_correlation', float('nan'))
        delta = (w - wo) if (w == w and wo == wo) else float('nan')  # NaN-safe
        print(f"  [{cond:>7}]  FC corr  with={w:.3f}  without={wo:.3f}  Δ={delta:+.3f}")

out_json = project_root / 'results' / 'lsd_drop_input_metrics.json'
out_json.parent.mkdir(parents=True, exist_ok=True)
out_json.write_text(json.dumps(
    {ckpt: {cond: {state: {kk: float(vv) for kk, vv in m.items()}
                   for state, m in cond_d.items()}
            for cond, cond_d in v.items()}
     for ckpt, v in results.items()},
    indent=2, sort_keys=True,
))
print(f'\nSaved: {out_json.relative_to(project_root)}')


=== lsd_hopf  (CoupledHopfModel, n_control_dims=2) ===
  [Placebo]  FC corr  with=0.393  without=0.391  Δ=+0.002
  [    LSD]  FC corr  with=0.963  without=0.964  Δ=-0.001
  [LSD+KET]  FC corr  with=0.882  without=0.880  Δ=+0.003

=== lsd_nsde  (NeuralSDE, n_control_dims=2) ===
  [Placebo]  FC corr  with=0.483  without=0.483  Δ=+0.000
  [    LSD]  FC corr  with=0.930  without=0.938  Δ=-0.007
  [LSD+KET]  FC corr  with=0.866  without=0.861  Δ=+0.005

=== lsd_hybrid_hopf  (HybridHopfModel, n_control_dims=2) ===
  [Placebo]  FC corr  with=0.350  without=0.429  Δ=-0.080
  [    LSD]  FC corr  with=0.957  without=0.961  Δ=-0.005
  [LSD+KET]  FC corr  with=0.885  without=0.874  Δ=+0.011

=== lsd_gnn_hopf  (GNNHopfModel, n_control_dims=2) ===
  [Placebo]  FC corr  with=0.507  without=0.507  Δ=+0.000
  [    LSD]  FC corr  with=0.982  without=0.986  Δ=-0.003
  [LSD+KET]  FC corr  with=0.854  without=0.850  Δ=+0.005

=== lsd_hybrid_neural  (HybridHopfNeuralModel, n_control_dims=0) ===
  [Placebo]

In [ ]:
def save(fig, name):
    fig.savefig(OUT_BASE / 'images' / 'comparison' / f'{name}.svg', dpi=200, bbox_inches='tight')
    fig.savefig(OUT_BASE / 'images_png' / 'comparison' / f'{name}.png', dpi=200, bbox_inches='tight')
    fig.savefig(OUT_BASE / 'images_svg' / 'comparison' / f'{name}.svg', bbox_inches='tight')


def get_value(metrics: dict, key: str):
    v = metrics.get(key, float('nan'))
    s = metrics.get(f'{key}_std', float('nan'))
    return float(v), float(s)


labels = [label for label, _, _ in METRICS]
x = np.arange(len(labels))
width = 0.36
panel_w = max(4.0, 0.6 * len(METRICS) + 0.5)

for ckpt_stem, (model_label, color) in MODELS.items():
    if ckpt_stem not in results:
        continue

    fig, axes = plt.subplots(
        1, len(CONDITIONS), figsize=(panel_w * len(CONDITIONS), 3.6), sharey=True,
    )
    fig.patch.set_alpha(0.0)

    for ax, (cond_name, _) in zip(axes, CONDITIONS):
        with_m = results[ckpt_stem][cond_name]['with']
        without_m = results[ckpt_stem][cond_name]['without']
        if not with_m or not without_m:
            ax.set_visible(False)
            continue

        with_vals = np.array([get_value(with_m, k)[0] for _, k, _ in METRICS])
        with_stds = np.array([get_value(with_m, k)[1] for _, k, _ in METRICS])
        without_vals = np.array([get_value(without_m, k)[0] for _, k, _ in METRICS])
        without_stds = np.array([get_value(without_m, k)[1] for _, k, _ in METRICS])

        ax.bar(x - width / 2, without_vals, width, yerr=without_stds, color=NO_INPUT_COLOR,
               edgecolor='none', capsize=3, label='No input',
               error_kw=dict(ecolor='black', lw=1.0))
        ax.bar(x + width / 2, with_vals, width, yerr=with_stds, color=color,
               edgecolor='none', capsize=3, label='Input',
               error_kw=dict(ecolor='black', lw=1.0))
        ax.set_xticks(x)
        ax.set_xticklabels(labels, fontsize=9, rotation=30, ha='right')
        ax.set_ylim(0, 1.0)
        ax.set_yticks([0.0, 0.5, 1.0])
        ax.set_title(cond_name, fontsize=12)
        ax.yaxis.grid(True, linestyle='--', alpha=0.5)
        ax.set_axisbelow(True)
        for spine in ('top', 'right'):
            ax.spines[spine].set_visible(False)

    axes[-1].legend(loc='upper right', bbox_to_anchor=(1.0, 1.0), frameon=False, fontsize=10)
    fig.suptitle(f'LSD — drop input ({model_label})', fontsize=14)
    plt.tight_layout()
    save(fig, f'lsd_drop_input_{ckpt_stem}')
    plt.show()
    plt.close(fig)

: 